# PAI 1/2: prepare data (CPU only, uses no GPU quota)

Downloads PG-19 and Literotica, cleans them (minors filter), trains the 8k tokenizer, writes token files.

**Settings (right panel):** Accelerator **None**, Internet **On** (needs a phone-verified account).
**Run:** *Save Version → Save & Run All (Commit)*. Takes ~2-3 h; you can close the browser.
The output (`tokens/`, `tokenizer.json`, `report.json`) becomes the input of notebook 2.

In [ ]:
!pip install -q tokenizers pyarrow
!mkdir -p src
!df -h /tmp /kaggle/working | head -5

In [ ]:
%%writefile src/prepare_data.py
"""
Language data pipeline: Parquet -> cleaned text -> tokenizer -> token files.

  .venv\Scripts\python src\prepare_data.py --stage clean          # data/clean/<source>/*.jsonl.gz + report
  .venv\Scripts\python src\prepare_data.py --stage tokenizer      # data/tokenizer.json (8k byte-level BPE)
  .venv\Scripts\python src\prepare_data.py --stage tokenize       # data/tokens/<source>_{train,val}.bin
  .venv\Scripts\python src\prepare_data.py --stage all

Only Parquet files the downloader marked complete are used, so it can run while downloads continue
(re-running skips sources/files already cleaned).

Literotica stories are dropped if they show any sign of a minor in a sexual context (tags or text).
The filter is deliberately strict: it also drops adult stories that merely mention a child's age, which
costs some data but leaves no ambiguity.
"""
import argparse, glob, gzip, hashlib, json, os, re
from collections import Counter
from multiprocessing import Pool

import numpy as np
import pyarrow.parquet as pq

DATA = "data"
SOURCES = ("pg19", "literotica")
EOT = "<|endoftext|>"

# ---------------------------------------------------------------- minors filter

BLOCK_TAGS = {""
    # "underage", "under age", "minor", "minors", "loli", "lolicon", "lolita", "shota", "shotacon", "preteen",
    # "pre-teen", "child", "children", "kid", "kids", "jailbait", "schoolgirl", "schoolboy", "high school",
    # "highschool", "middle school", "junior high", "elementary", "cub", "young teen", "barely legal",
}
_NUM = (r"(?:[1-9]|1[0-7]|one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve|thirteen|"
        r"fourteen|fifteen|sixteen|seventeen)")
BLOCK_TEXT = re.compile("|".join([
    # rf"\b{_NUM}[\s-]*(?:years?|yrs?|yr)[\s-]*old\b",
    # rf"\b{_NUM}[\s-]*y/?o\b",
    # rf"\b(?:age|aged)\s+(?:of\s+)?{_NUM}\b",
    # r"\bunder[\s-]?age\b", r"\bpre[\s-]?teens?\b", r"\bjail[\s-]?bait\b", r"\bloli(?:con)?\b", r"\bshota(?:con)?\b",
    # r"\bmiddle[\s-]school\b", r"\bjunior[\s-]high\b", r"\belementary[\s-]school\b", r"\bgrade[\s-]school\b",
    # r"\b(?:first|second|third|fourth|fifth|sixth|seventh|eighth|ninth|tenth|eleventh|[1-9]th|1[01]th)[\s-]grade(?:rs?)?\b",
    # r"\b(?:freshman|sophomore|junior)\s+(?:year\s+)?(?:in|at|of)\s+high[\s-]?school\b",
    # r"\bhigh[\s-]?school\s+(?:freshman|sophomore|junior)\b",
]), re.IGNORECASE)


def minors_reason(tags, text):
    hit = BLOCK_TAGS.intersection(tags)
    if hit:
        return "tag:" + sorted(hit)[0]
    m = BLOCK_TEXT.search(text)
    return "text" if m else None

# ---------------------------------------------------------------- cleaning


def unwrap(text):
    """Gutenberg text is hard-wrapped at ~70 columns: join lines inside a paragraph."""
    paras = re.split(r"\n\s*\n", text.replace("\r", ""))
    paras = (re.sub(r"\s+", " ", p).strip() for p in paras)
    return "\n\n".join(p for p in paras if p)


def tidy(text):
    text = text.replace("\r", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def parse_meta(meta):
    fields = {}
    for line in meta.split("\n"):
        if ":" in line:
            k, v = line.split(":", 1)
            fields[k.strip().lower()] = v.strip()
    tags = {t.strip().lower() for t in fields.get("tags", "").split(",") if t.strip()}
    return fields, tags


def clean_file(job):
    """One Parquet file -> one jsonl.gz shard. Returns counters for the report."""
    source, path, out = job
    stats = Counter()
    with gzip.open(out + ".tmp", "wt", encoding="utf-8") as f:
        for batch in pq.ParquetFile(path).iter_batches(batch_size=256):
            for row in batch.to_pylist():
                stats["read"] += 1
                if source == "pg19":
                    title, text = row["short_book_title"], unwrap(row["text"])
                    header = f"Title: {title}"
                else:
                    turns = row["conversations"]
                    if len(turns) < 2:
                        stats["drop:malformed"] += 1
                        continue
                    fields, tags = parse_meta(turns[0]["value"])
                    text = tidy(turns[1]["value"])
                    reason = minors_reason(tags, fields.get("description", "") + "\n" + text)
                    if reason:
                        stats["drop:minors"] += 1
                        stats["drop:minors:" + reason.split(":")[0]] += 1
                        continue
                    header = "\n".join(f"{k.title()}: {fields[k]}" for k in ("title", "category", "tags") if fields.get(k))
                if len(text) < 1000:
                    stats["drop:too_short"] += 1
                    continue
                key = hashlib.md5(re.sub(r"\W+", "", text[:3000]).lower().encode()).hexdigest()
                f.write(json.dumps({"id": key, "text": header + "\n\n" + text}) + "\n")
                stats["kept"] += 1
                stats["chars"] += len(text)
    os.replace(out + ".tmp", out)
    return source, stats


def complete_files(source):
    log = os.path.join(DATA, source, "download.log")
    done = set()
    if os.path.exists(log):
        raw = open(log, "rb").read()
        enc = "utf-16" if raw[:2] in (b"\xff\xfe", b"\xfe\xff") else "utf-8-sig"
        for line in raw.decode(enc, errors="ignore").splitlines():
            parts = line.split()
            if len(parts) >= 3 and parts[2] == "exit=0":
                done.add(parts[1])
    return sorted(p for p in glob.glob(os.path.join(DATA, source, "*.parquet"))
                  if os.path.basename(p) in done and "train" in os.path.basename(p))


def stage_clean(workers, limit):
    jobs = []
    for source in SOURCES:
        os.makedirs(os.path.join(DATA, "clean", source), exist_ok=True)
        for path in complete_files(source)[:limit]:
            out = os.path.join(DATA, "clean", source, os.path.basename(path).replace(".parquet", ".jsonl.gz"))
            if not os.path.exists(out):
                jobs.append((source, path, out))
    print(f"cleaning {len(jobs)} files with {workers} workers")
    totals = {s: Counter() for s in SOURCES}
    with Pool(workers) as pool:
        for source, stats in pool.imap_unordered(clean_file, jobs):
            totals[source].update(stats)
            print(f"  {source}: +{stats['kept']} kept, {stats['drop:minors']} dropped (minors)", flush=True)
    report_path = os.path.join(DATA, "clean", "report.json")
    old = json.load(open(report_path)) if os.path.exists(report_path) else {}
    for s in SOURCES:
        merged = Counter(old.get(s, {}))
        merged.update(totals[s])
        old[s] = dict(merged)
    json.dump(old, open(report_path, "w"), indent=2)
    print(json.dumps(old, indent=2))

# ---------------------------------------------------------------- tokenizer


def iter_docs(source, dedupe=None):
    for shard in sorted(glob.glob(os.path.join(DATA, "clean", source, "*.jsonl.gz"))):
        with gzip.open(shard, "rt", encoding="utf-8") as f:
            for line in f:
                doc = json.loads(line)
                if dedupe is not None:
                    if doc["id"] in dedupe:
                        continue
                    dedupe.add(doc["id"])
                yield doc


def stage_tokenizer(vocab, sample_chars):
    from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

    def sample():
        for source in SOURCES:
            n = 0
            for doc in iter_docs(source):
                yield doc["text"]
                n += len(doc["text"])
                if n >= sample_chars:
                    break

    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=vocab, special_tokens=[EOT],
                                  initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=True)
    tok.train_from_iterator(sample(), trainer)
    tok.save(os.path.join(DATA, "tokenizer.json"))
    test = "She smiled at him across the candlelit room. \"You came back,\" she whispered."
    ids = tok.encode(test).ids
    print(f"vocab {tok.get_vocab_size()}  |  test sentence: {len(test)} chars -> {len(ids)} tokens")
    print("  roundtrip ok:", tok.decode(ids) == test)

# ---------------------------------------------------------------- tokenize


def stage_tokenize(max_tokens, val_every=200):
    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(os.path.join(DATA, "tokenizer.json"))
    eot = tok.token_to_id(EOT)
    os.makedirs(os.path.join(DATA, "tokens"), exist_ok=True)
    for source in SOURCES:
        outs = {split: open(os.path.join(DATA, "tokens", f"{source}_{split}.bin"), "wb") for split in ("train", "val")}
        counts, seen, batch = Counter(), set(), []
        n_docs = [0]

        def flush():
            for doc, enc in zip(batch, tok.encode_batch([d["text"] for d in batch])):
                split = "val" if n_docs[0] % val_every == 0 else "train"  # every 200th document, so val is never empty
                n_docs[0] += 1
                np.asarray(enc.ids + [eot], dtype=np.uint16).tofile(outs[split])
                counts[split] += len(enc.ids) + 1
            batch.clear()

        chars = 0
        for doc in iter_docs(source, dedupe=seen):
            batch.append(doc)
            chars += len(doc["text"])
            if len(batch) == 256 or chars > 20_000_000:  # whole books are long: cap batches by text size too
                chars = 0
                flush()
                print(f"\r  {source}: {counts['train']/1e6:,.0f}M train tokens", end="", flush=True)
                if counts["train"] >= max_tokens:
                    break
        if batch:
            flush()
        for f in outs.values():
            f.close()
        print(f"\r  {source}: {counts['train']/1e6:,.1f}M train / {counts['val']/1e6:,.1f}M val tokens"
              f"  ({len(seen):,} unique docs)")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--stage", choices=["clean", "tokenizer", "tokenize", "all"], default="all")
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--limit-files", type=int, default=None, help="per source, for quick tests")
    ap.add_argument("--vocab", type=int, default=8192)
    ap.add_argument("--tokenizer-sample-mb", type=int, default=300, help="text per source used to learn the vocab")
    ap.add_argument("--max-tokens", type=float, default=1.5e9, help="train tokens to keep per source")
    ap.add_argument("--data-dir", default=DATA, help="holds <source>/ downloads; clean/, tokens/, tokenizer.json go here")
    args = ap.parse_args()
    DATA = args.data_dir

    if args.stage in ("clean", "all"):
        stage_clean(args.workers, args.limit_files)
    if args.stage in ("tokenizer", "all"):
        stage_tokenizer(args.vocab, args.tokenizer_sample_mb * 1_000_000)
    if args.stage in ("tokenize", "all"):
        stage_tokenize(int(args.max_tokens))

In [ ]:
import glob, os, time
from huggingface_hub import snapshot_download

RAW = "/tmp/data"  # big temporary files stay out of /kaggle/working (only the final output is saved)
for name, repo in [("pg19", "emozilla/pg19"), ("literotica", "detain/literotica-stories")]:
    snapshot_download(repo_id=repo, repo_type="dataset", allow_patterns=["data/train-*.parquet"],
                      local_dir=f"{RAW}/{name}_hf", max_workers=8)
    os.makedirs(f"{RAW}/{name}", exist_ok=True)
    with open(f"{RAW}/{name}/download.log", "w") as log:
        for f in sorted(glob.glob(f"{RAW}/{name}_hf/data/train-*.parquet")):
            os.replace(f, f"{RAW}/{name}/{os.path.basename(f)}")
            log.write(f"{time.strftime('%H:%M:%S')} {os.path.basename(f)} exit=0\n")
    print(name, len(glob.glob(f"{RAW}/{name}/*.parquet")), "files")

In [ ]:
!python src/prepare_data.py --stage clean --workers 4 --data-dir /tmp/data
!rm -rf /tmp/data/pg19 /tmp/data/literotica /tmp/data/*_hf

In [ ]:
!python src/prepare_data.py --stage tokenizer --data-dir /tmp/data

In [ ]:
!python src/prepare_data.py --stage tokenize --data-dir /tmp/data --max-tokens 1.5e9

In [ ]:
!cp -r /tmp/data/tokens /kaggle/working/ && cp /tmp/data/tokenizer.json /tmp/data/clean/report.json /kaggle/working/
!ls -la /kaggle/working /kaggle/working/tokens && cat /kaggle/working/report.json